# FIRE Kenya — Comprehensive Financial Independence Analysis

**A rigorous computational study of the Financial Independence, Retire Early (FIRE) framework, calibrated for the Kenyan economic context.**

This notebook derives, implements, and validates every mathematical formula used in the FIRE Kenya dashboard. It is structured as a self-contained reference covering:

| Module | Content |
|---|---|
| **§1** | Savings Rate & FIRE Number derivation |
| **§2** | Compound interest, inflation adjustment, and portfolio projection |
| **§3** | Future Value of Annuity & Reverse FIRE engineering |
| **§4** | Fee drag computation (ETF vs active fund) |
| **§5** | Currency hedge return (KES equivalent of USD assets) |
| **§6** | Vehicle ownership: fuel + financing opportunity cost |
| **§7** | Transport expenditure impact (KES 1,000/week) |
| **§8** | Insurance & education cost forecasting |
| **§9** | Scenario analysis & sensitivity tables |
| **§10** | Full 27-year trajectory simulation |

---

### Baseline Parameters

| Parameter | Value | Source |
|---|---|---|
| Current Age | 23 | User profile |
| Target Retirement Age | 50 | User goal |
| Years to FIRE | 27 | Derived |
| Current Savings | KES 173,000 | User profile |
| Current Stipend | KES 15,480/month | User profile |
| Phase 2 Salary | KES 59,000/month | Sep 2026 – Jun 2028 |
| Savings Rate | 30% | 50/30/20 budget framework |
| Expected Nominal Return | 12% p.a. | Blended Kenyan portfolio |
| Inflation | 6% p.a. | Kenya long-run average |
| Safe Withdrawal Rate | 3.5% | Conservative (40+ year horizon) |
| Family Size | 5 | Self + spouse + 3 children |
| Weekly Transport | KES 1,000 | Fixed operating cost |

In [ ]:
# ============================================================
# SETUP: Imports and formatting utilities
# ============================================================

import math
from typing import Tuple, List

def fmt(amount: float) -> str:
    """Format a number as KES currency."""
    return f"KES {amount:,.0f}"

def pct(value: float) -> str:
    """Format a decimal as a percentage string."""
    return f"{value * 100:.2f}%"

# Baseline constants (mirroring PERSONAL config in app.js)
CURRENT_AGE        = 23
RETIRE_AGE         = 50
YEARS_TO_FIRE      = RETIRE_AGE - CURRENT_AGE  # 27
MONTHS_TO_FIRE     = YEARS_TO_FIRE * 12         # 324

CURRENT_SAVINGS    = 173_000
STIPEND            = 15_480
JOB_SALARY         = 59_000
SAVINGS_RATE       = 0.30

NOMINAL_RETURN     = 0.12
INFLATION          = 0.06
SWR                = 0.035

WEEKLY_TRANSPORT   = 1_000
MONTHLY_TRANSPORT  = round(WEEKLY_TRANSPORT * 52 / 12)  # 4,333
ANNUAL_TRANSPORT   = WEEKLY_TRANSPORT * 52               # 52,000

CHILDREN           = 3
FAMILY_SIZE        = 5

print(f"Horizon: {YEARS_TO_FIRE} years ({MONTHS_TO_FIRE} months)")
print(f"Monthly transport: {fmt(MONTHLY_TRANSPORT)}")
print(f"Annual transport:  {fmt(ANNUAL_TRANSPORT)}")

---

## §1 — Savings Rate & FIRE Number

### 1.1 Savings Rate

The savings rate is the fraction of income directed to wealth accumulation:

$$SR = \frac{\text{Income} - \text{Expenses}}{\text{Income}} = \frac{\text{Savings}}{\text{Income}}$$

Under the **50/30/20 budget framework**:
- 50% → Essentials (rent, food, transport, utilities)
- 30% → Investments
- 20% → Discretionary / Entertainment

### 1.2 FIRE Number

The **FIRE Number** is the portfolio value at which annual investment income sustains retirement expenses indefinitely. Derived from the Safe Withdrawal Rate:

$$\text{FIRE Number} = \frac{\text{Annual Retirement Expenses}}{\text{SWR}}$$

With inflation adjustment over the accumulation horizon:

$$\text{FIRE Number} = \frac{E_{\text{annual}} \times (1 + i)^{t}}{\text{SWR}}$$

where:
- $E_{\text{annual}}$ = current annual expenses (today's value)
- $i$ = annual inflation rate
- $t$ = years to retirement
- $\text{SWR}$ = safe withdrawal rate

| SWR | Multiplier | Suitable For |
|---|---|---|
| 4.0% (Trinity Study) | 25× | 30-year horizons, low-inflation economies |
| 3.5% (Conservative) | 28.57× | 40+ year horizons, volatile/high-inflation regions |
| 3.0% (Ultra-safe) | 33.33× | Maximum longevity protection |

In [ ]:
# ============================================================
# §1: Savings Rate & FIRE Number
# ============================================================

def savings_rate(income: float, expenses: float) -> float:
    """Calculate savings rate as a decimal (0.0 – 1.0)."""
    if income <= 0:
        raise ValueError("Income must be positive.")
    return (income - expenses) / income


def fire_number(annual_expenses: float, inflation: float,
                years: int, swr: float) -> Tuple[float, float]:
    """Calculate inflation-adjusted FIRE number.
    
    Returns:
        (inflated_annual_expenses, fire_number)
    """
    inflated = annual_expenses * (1 + inflation) ** years
    return inflated, inflated / swr


# --- Stipend Phase ---
stipend_savings = STIPEND * SAVINGS_RATE
stipend_expenses = STIPEND - stipend_savings
sr_stipend = savings_rate(STIPEND, stipend_expenses)

print("=" * 60)
print("§1: SAVINGS RATE & FIRE NUMBER")
print("=" * 60)
print(f"Stipend:           {fmt(STIPEND)}/month")
print(f"Monthly savings:   {fmt(stipend_savings)} (SR: {pct(sr_stipend)})")
print(f"Monthly expenses:  {fmt(stipend_expenses)}")
print()

# --- FIRE Number at different SWRs ---
target_monthly_expenses = 100_000  # KES 100K/month today
target_annual = target_monthly_expenses * 12

print(f"Target retirement lifestyle: {fmt(target_monthly_expenses)}/month (today's value)")
print(f"Annual equivalent:           {fmt(target_annual)}")
print()

for rate in [0.04, 0.035, 0.03]:
    inflated_exp, fn = fire_number(target_annual, INFLATION, YEARS_TO_FIRE, rate)
    print(f"  SWR {rate*100:.1f}%: Inflated expenses = {fmt(inflated_exp)}/yr"
          f" → FIRE Number = {fmt(fn)}")

# Primary target (3.5% SWR)
inflated_annual, FIRE_TARGET = fire_number(target_annual, INFLATION, YEARS_TO_FIRE, SWR)
print(f"\n★ Primary FIRE Target (3.5% SWR): {fmt(FIRE_TARGET)}")
print(f"  = {fmt(inflated_annual)}/yr inflated expenses ÷ {SWR}")
print(f"  = {fmt(target_monthly_expenses)}/mo × 12 × (1.06)^{YEARS_TO_FIRE} ÷ 0.035")

---

## §2 — Compound Interest, Inflation & Portfolio Projection

### 2.1 Future Value with Monthly Contributions

The future value of a portfolio receiving regular monthly contributions, with monthly compounding:

$$FV = PV \times (1 + r_m)^n + PMT \times \frac{(1 + r_m)^n - 1}{r_m}$$

where:
- $PV$ = present value (initial lump sum)
- $PMT$ = monthly contribution
- $r_m = r_{\text{annual}} / 12$ = monthly rate
- $n = t \times 12$ = total months

### 2.2 Nominal vs Real Returns

The **Fisher Equation** adjusts nominal returns for inflation to express growth in today's purchasing power:

$$r_{\text{real}} = \frac{1 + r_{\text{nominal}}}{1 + i} - 1$$

### 2.3 Real Portfolio Value

To express the nominal future value in today's purchasing power:

$$FV_{\text{real}}(t) = \frac{FV_{\text{nominal}}(t)}{(1 + i)^t}$$

In [ ]:
# ============================================================
# §2: Portfolio Projection Engine
# ============================================================

def project_portfolio(
    initial: float,
    monthly_contrib: float,
    annual_return: float,
    inflation: float,
    years: int
) -> Tuple[List[float], List[float]]:
    """Month-by-month portfolio projection.
    
    Returns:
        (nominal_history, real_history) — lists of month-end balances.
    """
    r_m = annual_return / 12
    months = years * 12
    
    nominal = []
    real = []
    balance = initial
    
    for m in range(1, months + 1):
        balance = balance * (1 + r_m) + monthly_contrib
        nominal.append(balance)
        real.append(balance / (1 + inflation) ** (m / 12))
    
    return nominal, real


def fv_annuity(pv: float, pmt: float, r_annual: float, years: int) -> float:
    """Closed-form future value: lump sum + annuity (monthly compounding)."""
    r_m = r_annual / 12
    n = years * 12
    return pv * (1 + r_m) ** n + pmt * ((1 + r_m) ** n - 1) / r_m


def real_return(nominal: float, inflation: float) -> float:
    """Fisher equation: inflation-adjusted return."""
    return (1 + nominal) / (1 + inflation) - 1


# --- Projection: Stipend phase savings over 27 years ---
monthly_invest_stipend = STIPEND * SAVINGS_RATE  # KES 4,644

nom_hist, real_hist = project_portfolio(
    CURRENT_SAVINGS, monthly_invest_stipend,
    NOMINAL_RETURN, INFLATION, YEARS_TO_FIRE
)

# Closed-form verification
fv_closed = fv_annuity(CURRENT_SAVINGS, monthly_invest_stipend, NOMINAL_RETURN, YEARS_TO_FIRE)
r_real = real_return(NOMINAL_RETURN, INFLATION)

print("=" * 60)
print("§2: PORTFOLIO PROJECTION (constant KES 4,644/mo contribution)")
print("=" * 60)
print(f"Initial savings:     {fmt(CURRENT_SAVINGS)}")
print(f"Monthly investment:  {fmt(monthly_invest_stipend)}")
print(f"Nominal return:      {pct(NOMINAL_RETURN)} p.a.")
print(f"Inflation:           {pct(INFLATION)} p.a.")
print(f"Real return (Fisher): {pct(r_real)} p.a.")
print()
print(f"Nominal FV (iterative):   {fmt(nom_hist[-1])}")
print(f"Nominal FV (closed-form): {fmt(fv_closed)}")
print(f"Real FV (today's KES):    {fmt(real_hist[-1])}")
print()

# Milestone checkpoints
print("Year-by-year milestones (nominal):")
for y in [1, 2, 5, 10, 15, 20, 25, 27]:
    idx = min(y * 12 - 1, len(nom_hist) - 1)
    age = CURRENT_AGE + y
    print(f"  Year {y:2d} (Age {age}): {fmt(nom_hist[idx]):>20s}")

---

## §3 — Reverse FIRE Engineering

### 3.1 Solving for Required Monthly Savings

Given a FIRE target ($FV$), current savings ($PV$), expected return ($r$), and horizon ($n$ months), solve the FV annuity equation for $PMT$:

$$PMT = \frac{FV - PV \times (1 + r_m)^n}{\frac{(1 + r_m)^n - 1}{r_m}}$$

### 3.2 Solving for Required Return (Newton-Raphson)

When the monthly savings is fixed and we need to find the return rate that reaches the FIRE target, define:

$$f(r) = PMT \times \frac{(1+r)^n - 1}{r} + PV \times (1+r)^n - FV$$

$$f'(r) = PMT \times \frac{n(1+r)^{n-1} \cdot r - (1+r)^n + 1}{r^2} + PV \cdot n \cdot (1+r)^{n-1}$$

Iterate: $r_{i+1} = r_i - \frac{f(r_i)}{f'(r_i)}$ until $|f(r_i)| < \epsilon$

In [ ]:
# ============================================================
# §3: Reverse FIRE Engineering
# ============================================================

def required_monthly_savings(
    fire_target: float, current_savings: float,
    annual_return: float, years: int
) -> float:
    """Solve for PMT given a target FV."""
    r_m = annual_return / 12
    n = years * 12
    pv_grown = current_savings * (1 + r_m) ** n
    annuity_factor = ((1 + r_m) ** n - 1) / r_m
    return max(0, (fire_target - pv_grown) / annuity_factor)


def required_return_newton(
    fire_target: float, current_savings: float,
    monthly_savings: float, months: int,
    max_iter: int = 100, tol: float = 1_000
) -> float:
    """Newton-Raphson solve for monthly rate, returns annualized rate."""
    r = 0.01  # initial guess (monthly)
    
    for _ in range(max_iter):
        comp = (1 + r) ** months
        annuity = (comp - 1) / r
        f = monthly_savings * annuity + current_savings * comp - fire_target
        
        # Derivative
        d_comp = months * (1 + r) ** (months - 1)
        d_annuity = (d_comp * r - comp + 1) / (r ** 2)
        f_prime = monthly_savings * d_annuity + current_savings * d_comp
        
        if abs(f_prime) < 1e-12:
            break
        r -= f / f_prime
        r = max(r, 1e-8)  # prevent negative rates
        
        if abs(f) < tol:
            break
    
    return (1 + r) ** 12 - 1


print("=" * 60)
print("§3: REVERSE FIRE ENGINEERING")
print("=" * 60)

req_pmt = required_monthly_savings(FIRE_TARGET, CURRENT_SAVINGS, NOMINAL_RETURN, YEARS_TO_FIRE)
print(f"FIRE Target:                {fmt(FIRE_TARGET)}")
print(f"Required monthly savings:   {fmt(req_pmt)}")
print(f"  as % of KES 59,000 salary: {req_pmt / JOB_SALARY * 100:.1f}%")
print()

# Maximum affordable monthly expenditure
max_expense = JOB_SALARY - req_pmt
print(f"Max monthly expenditure:    {fmt(max(0, max_expense))}")
print()

# Required return at current savings rate
current_monthly_saving = STIPEND * SAVINGS_RATE
req_return = required_return_newton(
    FIRE_TARGET, CURRENT_SAVINGS,
    current_monthly_saving, MONTHS_TO_FIRE
)
print(f"Required return (at {fmt(current_monthly_saving)}/mo): {pct(req_return)} p.a.")
print(f"  (Newton-Raphson converged to monthly rate {req_return/12*100:.4f}%)")

# Verification: plug required PMT back into FV
fv_check = fv_annuity(CURRENT_SAVINGS, req_pmt, NOMINAL_RETURN, YEARS_TO_FIRE)
print(f"\nVerification: FV with {fmt(req_pmt)}/mo = {fmt(fv_check)}")
print(f"  Matches FIRE target? {'✓' if abs(fv_check - FIRE_TARGET) < 1000 else '✗'}")

---

## §4 — Fee Drag: ETF vs Active Fund

Fund management fees compound *against* the investor. The **fee drag** is the wealth destroyed by higher fees over time:

$$\text{Net Return}_{\text{fund}} = r_{\text{gross}} - f_{\text{fund}}$$

$$FV_{\text{fund}} = P \times (1 + r_{\text{gross}} - f_{\text{fund}})^{t}$$

$$\text{Fee Drag} = FV_{\text{low-cost}} - FV_{\text{high-cost}}$$

$$\text{Drag \%} = \frac{\text{Fee Drag}}{FV_{\text{low-cost}}} \times 100$$

### Historical Performance of Major Global Indices

| Index / ETF | Ticker | Return (USD p.a.) | Period | Expense Ratio | Benchmark |
|---|---|---|---|---|---|
| S&P 500 | VOO / SPY | 10.5% | 1957–2025 | 0.03% | S&P 500 Index |
| MSCI World | IWDA | 8.8% | 1987–2025 | 0.20% | MSCI World Index |
| MSCI Emerging Markets | VWO / EEM | 9.2% | 2000–2025 | 0.08% | MSCI EM Index |
| Nasdaq-100 | QQQ | 14.2% | 2000–2025 | 0.20% | Nasdaq-100 Index |
| FTSE All-World | VWRA | 8.5% | 2005–2025 | 0.22% | FTSE All-World |
| Global Agg Bond | BND / AGGU | 4.1% | 2007–2025 | 0.03% | Bloomberg Agg |

### Recommended ETF Allocation (Growth Phase)

| Category | ETF | Weight | Expense Ratio | Rationale |
|---|---|---|---|---|
| Core Equities | VOO | 50% | 0.03% | US large-cap, ultra-low cost |
| Global Developed | IWDA | 25% | 0.20% | 23 developed markets |
| Emerging Markets | VWO | 15% | 0.08% | China, India, Brazil, Africa |
| Fixed Income | AGGU | 10% | 0.10% | Volatility buffer |

**Rebalancing**: Annually or when any position drifts ±5%. Redirect new contributions to underweight assets.

**Tax (Kenya)**: US dividend withholding 30% (no treaty); Kenyan CGT 15%; use Ireland-domiciled accumulating ETFs (IWDA, VWRA) to defer taxes.

In [ ]:
# ============================================================
# §4: Fee Drag Computation
# ============================================================

def fee_drag(
    principal: float, gross_return: float,
    fee_low: float, fee_high: float, years: int
) -> Tuple[float, float, float, float]:
    """Calculate fee drag between two fee levels.
    
    Returns:
        (fv_low, fv_high, drag_kes, drag_pct)
    """
    fv_low = principal * (1 + gross_return - fee_low) ** years
    fv_high = principal * (1 + gross_return - fee_high) ** years
    drag = fv_low - fv_high
    drag_pct = (drag / fv_low) * 100 if fv_low > 0 else 0
    return fv_low, fv_high, drag, drag_pct


print("=" * 60)
print("§4: FEE DRAG — ETF vs ACTIVE FUND")
print("=" * 60)

etf_fee = 0.0003     # 0.03% (Vanguard VOO)
active_fee = 0.0200  # 2.00% (typical Kenyan unit trust)
principal = 100_000
gross = 0.12

fv_etf, fv_active, drag_kes, drag_pct = fee_drag(
    principal, gross, etf_fee, active_fee, YEARS_TO_FIRE
)

print(f"Principal:         {fmt(principal)}")
print(f"Gross return:      {pct(gross)} p.a.")
print(f"ETF fee:           {etf_fee*100:.2f}% → net {pct(gross - etf_fee)}")
print(f"Active fund fee:   {active_fee*100:.2f}% → net {pct(gross - active_fee)}")
print(f"Horizon:           {YEARS_TO_FIRE} years")
print()
print(f"ETF terminal value:        {fmt(fv_etf)}")
print(f"Active fund terminal value: {fmt(fv_active)}")
print(f"Fee drag (wealth lost):    {fmt(drag_kes)} ({drag_pct:.1f}%)")
print()

# Sensitivity: fee drag at different return levels
print("Fee drag sensitivity (KES 100K, 27 years, ETF 0.03% vs Active 2.0%):")
print(f"{'Gross Return':>14s} {'ETF Yield':>16s} {'Active Yield':>16s} {'Drag':>16s} {'Drag %':>8s}")
print("-" * 72)
for r in [0.08, 0.10, 0.12, 0.14, 0.16, 0.18]:
    fv_e, fv_a, d, dp = fee_drag(principal, r, etf_fee, active_fee, YEARS_TO_FIRE)
    print(f"{r*100:>13.0f}% {fmt(fv_e):>16s} {fmt(fv_a):>16s} {fmt(d):>16s} {dp:>7.1f}%")

---

## §5 — Currency Hedge Return

For Kenyan investors holding USD-denominated ETFs, the effective KES return incorporates currency depreciation:

$$r_{\text{KES}} = (1 + r_{\text{USD}}) \times (1 + d_{\text{KES/USD}}) - 1$$

where $d_{\text{KES/USD}}$ = annual KES depreciation rate against USD (~3–5% historically).

This means a 10.5% USD return on S&P 500 effectively yields **~13.5–15.5%** in KES terms — a structural advantage of holding foreign-currency assets for Kenyan FIRE investors.

In [ ]:
# ============================================================
# §5: Currency Hedge Return
# ============================================================

def kes_equivalent_return(usd_return: float, kes_depreciation: float) -> float:
    """Convert USD return to KES-equivalent return."""
    return (1 + usd_return) * (1 + kes_depreciation) - 1


print("=" * 60)
print("§5: CURRENCY HEDGE — USD TO KES RETURN")
print("=" * 60)

indices = [
    ("S&P 500 (VOO)",      0.105),
    ("MSCI World (IWDA)",   0.088),
    ("MSCI EM (VWO)",       0.092),
    ("Nasdaq-100 (QQQ)",    0.142),
    ("FTSE All-World",      0.085),
    ("Global Bonds (AGGU)", 0.041),
]

print(f"{'Index':<24s} {'USD Return':>10s} {'KES @3%':>10s} {'KES @4%':>10s} {'KES @5%':>10s}")
print("-" * 66)
for name, usd_r in indices:
    k3 = kes_equivalent_return(usd_r, 0.03)
    k4 = kes_equivalent_return(usd_r, 0.04)
    k5 = kes_equivalent_return(usd_r, 0.05)
    print(f"{name:<24s} {usd_r*100:>9.1f}% {k3*100:>9.1f}% {k4*100:>9.1f}% {k5*100:>9.1f}%")

---

## §6 — Vehicle Ownership: Fuel & Financing Opportunity Cost

A car with **KES 7,000/month financing** and **KES 7,000/month fuel** creates KES 14,000/month in cash outflow. Under a KES 59,000 salary with 30% savings rate (KES 17,700/month), the vehicle consumes 79% of investment capacity:

| Scenario | Monthly Savings | Reduction |
|---|---|---|
| A: No car (baseline) | KES 17,700 | — |
| B: Car owned outright (fuel only) | KES 10,700 | −39.5% |
| C: Financed car + fuel | KES 3,700 | −79.1% |

The opportunity cost is the future value of the diverted savings:

$$\text{Opportunity Cost} = PMT_{\text{diverted}} \times \frac{(1 + r_m)^n - 1}{r_m}$$

In [ ]:
# ============================================================
# §6: Vehicle Ownership Opportunity Cost
# ============================================================

def opportunity_cost(monthly_diversion: float, annual_return: float, years: int) -> float:
    """FV of a monthly cash flow diverted from investments."""
    r_m = annual_return / 12
    n = years * 12
    return monthly_diversion * ((1 + r_m) ** n - 1) / r_m


def years_to_target(
    initial: float, monthly: float,
    target: float, annual_return: float, max_months: int = 1200
) -> float:
    """Iteratively find months to reach a target balance."""
    r_m = annual_return / 12
    balance = initial
    for m in range(1, max_months + 1):
        balance = balance * (1 + r_m) + monthly
        if balance >= target:
            return m / 12
    return max_months / 12


print("=" * 60)
print("§6: VEHICLE OWNERSHIP OPPORTUNITY COST")
print("=" * 60)

fuel_cost = 7_000
financing_cost = 7_000
total_car = fuel_cost + financing_cost

baseline_savings = JOB_SALARY * SAVINGS_RATE  # 17,700
savings_fuel_only = baseline_savings - fuel_cost
savings_full_car = baseline_savings - total_car

# Opportunity costs
oc_fuel = opportunity_cost(fuel_cost, NOMINAL_RETURN, YEARS_TO_FIRE)
oc_finance = opportunity_cost(financing_cost, NOMINAL_RETURN, YEARS_TO_FIRE)
oc_total = oc_fuel + oc_finance

print(f"Phase 2 salary:          {fmt(JOB_SALARY)}/month")
print(f"Baseline savings (30%):  {fmt(baseline_savings)}/month")
print(f"Fuel cost:               {fmt(fuel_cost)}/month")
print(f"Financing cost:          {fmt(financing_cost)}/month")
print()
print(f"Opportunity cost of fuel (27 yrs @ 12%):       {fmt(oc_fuel)}")
print(f"Opportunity cost of financing (27 yrs @ 12%):  {fmt(oc_finance)}")
print(f"Total opportunity cost:                        {fmt(oc_total)}")
print()

# Portfolio comparison at age 50
nom_a, _ = project_portfolio(CURRENT_SAVINGS, baseline_savings, NOMINAL_RETURN, INFLATION, YEARS_TO_FIRE)
nom_b, _ = project_portfolio(CURRENT_SAVINGS, savings_fuel_only, NOMINAL_RETURN, INFLATION, YEARS_TO_FIRE)
nom_c, _ = project_portfolio(CURRENT_SAVINGS, savings_full_car, NOMINAL_RETURN, INFLATION, YEARS_TO_FIRE)

print("PORTFOLIO AT AGE 50 (nominal):")
print(f"  A — No car:                    {fmt(nom_a[-1])}")
print(f"  B — Fuel only (KES 7K/mo):     {fmt(nom_b[-1])}")
print(f"  C — Finance + fuel (KES 14K):  {fmt(nom_c[-1])}")
print()

# Retirement delay
target_portfolio = nom_a[-1]
t_b = years_to_target(CURRENT_SAVINGS, savings_fuel_only, target_portfolio, NOMINAL_RETURN)
t_c = years_to_target(CURRENT_SAVINGS, savings_full_car, target_portfolio, NOMINAL_RETURN)

print("RETIREMENT DELAY:")
print(f"  A — Baseline:        {YEARS_TO_FIRE:.0f} years (retire at 50)")
print(f"  B — Fuel only:       {t_b:.1f} years (delay +{t_b - YEARS_TO_FIRE:.1f}, retire at {CURRENT_AGE + t_b:.1f})")
print(f"  C — Finance + fuel:  {t_c:.1f} years (delay +{t_c - YEARS_TO_FIRE:.1f}, retire at {CURRENT_AGE + t_c:.1f})")

---

## §7 — Transport Expenditure Impact

Fixed weekly transport of **KES 1,000** (monthly KES 4,333, annual KES 52,000). This section quantifies the compound opportunity cost and its effect on the FIRE timeline.

In [ ]:
# ============================================================
# §7: Transport Expenditure Impact
# ============================================================

print("=" * 60)
print("§7: TRANSPORT COST IMPACT (KES 1,000/week)")
print("=" * 60)

print(f"Weekly:   {fmt(WEEKLY_TRANSPORT)}")
print(f"Monthly:  {fmt(MONTHLY_TRANSPORT)}")
print(f"Annual:   {fmt(ANNUAL_TRANSPORT)}")
print()

oc_transport = opportunity_cost(MONTHLY_TRANSPORT, NOMINAL_RETURN, YEARS_TO_FIRE)
print(f"27-year opportunity cost (@ 12%): {fmt(oc_transport)}")
print()

# With vs without transport deducted from savings
savings_with_transport = baseline_savings - MONTHLY_TRANSPORT
nom_transport, _ = project_portfolio(
    CURRENT_SAVINGS, savings_with_transport,
    NOMINAL_RETURN, INFLATION, YEARS_TO_FIRE
)
nom_no_transport, _ = project_portfolio(
    CURRENT_SAVINGS, baseline_savings,
    NOMINAL_RETURN, INFLATION, YEARS_TO_FIRE
)

print(f"Portfolio at 50 (no transport):   {fmt(nom_no_transport[-1])}")
print(f"Portfolio at 50 (with transport): {fmt(nom_transport[-1])}")
print(f"Difference:                      {fmt(nom_no_transport[-1] - nom_transport[-1])}")

t_transport = years_to_target(
    CURRENT_SAVINGS, savings_with_transport,
    nom_no_transport[-1], NOMINAL_RETURN
)
print(f"Retirement delay due to transport: +{t_transport - YEARS_TO_FIRE:.1f} years")

---

## §8 — Insurance & Education Forecasting

### 8.1 Insurance Premiums by Phase

| Type | Monthly | From | Annual |
|---|---|---|---|
| NHIF / SHA | KES 1,700 | Employment start | KES 20,400 |
| Private Health | KES 5,000 | Age 28 | KES 60,000 |
| Car Insurance | KES 5,208 | Age 28 (5% of KES 1.25M) | KES 62,500 |
| Life Insurance | KES 3,000 | Age 30 | KES 36,000 |

### 8.2 Education Cost Model

$$\text{Tuition}(y) = \text{Base Tuition} \times (1 + i_e)^{(y - y_{\text{birth}} - a_{\text{entry}})}$$

where $i_e = 8\%$ (education inflation, Kenya).

| Level | Entry Age | Duration | Base Annual Fee |
|---|---|---|---|
| Primary | 6 | 8 years | KES 150,000 |
| Secondary | 14 | 4 years | KES 250,000 |
| University | 18 | 4 years | KES 400,000 |

In [ ]:
# ============================================================
# §8: Insurance & Education Forecasting
# ============================================================

def total_insurance_cost(years_employed: int) -> dict:
    """Calculate cumulative insurance costs over career."""
    nhif_annual = 1_700 * 12
    private_annual = 5_000 * 12
    car_annual = 0.05 * 1_250_000
    life_annual = 3_000 * 12
    
    total = 0
    breakdown = {"NHIF": 0, "Private Health": 0, "Car": 0, "Life": 0}
    
    for yr in range(years_employed):
        age = CURRENT_AGE + yr
        if age >= 23:  # employed
            breakdown["NHIF"] += nhif_annual
        if age >= 28:
            breakdown["Private Health"] += private_annual
            breakdown["Car"] += car_annual
        if age >= 30:
            breakdown["Life"] += life_annual
    
    breakdown["Total"] = sum(v for k, v in breakdown.items())
    return breakdown


def total_education_cost(
    birth_years: list, edu_inflation: float = 0.08
) -> Tuple[float, dict]:
    """Total education cost for multiple children."""
    levels = [
        ("Primary",   6, 8, 150_000),
        ("Secondary", 14, 4, 250_000),
        ("University", 18, 4, 400_000),
    ]
    total = 0
    details = {}
    
    for i, birth_yr in enumerate(birth_years):
        child_total = 0
        for level_name, entry_age, duration, base_fee in levels:
            for yr_offset in range(duration):
                cal_year = birth_yr + entry_age + yr_offset
                years_inflated = cal_year - 2026  # inflation from now
                inflated_fee = base_fee * (1 + edu_inflation) ** max(0, years_inflated)
                child_total += inflated_fee
        details[f"Child {i+1} (born {birth_yr})"] = child_total
        total += child_total
    
    return total, details


print("=" * 60)
print("§8: INSURANCE & EDUCATION FORECASTING")
print("=" * 60)

# Insurance
ins = total_insurance_cost(YEARS_TO_FIRE)
print("CUMULATIVE INSURANCE COSTS (Age 23 → 50):")
for k, v in ins.items():
    print(f"  {k:<18s} {fmt(v)}")
print()

# Education
birth_years = [2032, 2034, 2036]
edu_total, edu_details = total_education_cost(birth_years)
print(f"EDUCATION COSTS (8% inflation, {CHILDREN} children):")
for k, v in edu_details.items():
    print(f"  {k:<26s} {fmt(v)}")
print(f"  {'TOTAL':<26s} {fmt(edu_total)}")
print()

# Impact: education + insurance as fraction of FIRE target
combined = ins['Total'] + edu_total
print(f"Combined insurance + education: {fmt(combined)}")
print(f"As % of FIRE target ({fmt(FIRE_TARGET)}): {combined / FIRE_TARGET * 100:.1f}%")

---

## §9 — Scenario Analysis & Sensitivity Tables

How does the terminal portfolio respond to changes in return rate, savings rate, and inflation?

In [ ]:
# ============================================================
# §9: Scenario Analysis & Sensitivity
# ============================================================

print("=" * 60)
print("§9: SENSITIVITY ANALYSIS")
print("=" * 60)

# 9.1: Return rate sensitivity (fixed KES 17,700/mo savings)
print("\n9.1 RETURN RATE SENSITIVITY")
print(f"  (KES 173K initial, KES 17,700/mo, {YEARS_TO_FIRE} years)")
print(f"  {'Return':>8s} {'Nominal FV':>18s} {'Real FV':>18s} {'Hits FIRE?':>12s}")
print("  " + "-" * 58)
for r in [0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18]:
    fv_nom = fv_annuity(CURRENT_SAVINGS, baseline_savings, r, YEARS_TO_FIRE)
    fv_real_val = fv_nom / (1 + INFLATION) ** YEARS_TO_FIRE
    hit = "✓" if fv_nom >= FIRE_TARGET else "✗"
    print(f"  {r*100:>7.0f}% {fmt(fv_nom):>18s} {fmt(fv_real_val):>18s} {hit:>12s}")

# 9.2: Savings rate sensitivity (fixed 12% return)
print("\n9.2 SAVINGS RATE SENSITIVITY")
print(f"  (KES 59,000 salary, 12% return, {YEARS_TO_FIRE} years)")
print(f"  {'SR':>8s} {'Monthly':>12s} {'Nominal FV':>18s} {'Hits FIRE?':>12s}")
print("  " + "-" * 52)
for sr in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pmt = JOB_SALARY * sr
    fv_nom = fv_annuity(CURRENT_SAVINGS, pmt, NOMINAL_RETURN, YEARS_TO_FIRE)
    hit = "✓" if fv_nom >= FIRE_TARGET else "✗"
    print(f"  {sr*100:>7.0f}% {fmt(pmt):>12s} {fmt(fv_nom):>18s} {hit:>12s}")

# 9.3: Inflation sensitivity (fixed 12% nominal, KES 17,700/mo)
print("\n9.3 INFLATION SENSITIVITY")
print(f"  (Impact on FIRE number at 3.5% SWR, KES 100K/mo target)")
print(f"  {'Inflation':>10s} {'Inflated Exp':>16s} {'FIRE Number':>18s} {'Real Return':>14s}")
print("  " + "-" * 60)
for inf in [0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.10]:
    infl_exp, fn = fire_number(target_annual, inf, YEARS_TO_FIRE, SWR)
    rr = real_return(NOMINAL_RETURN, inf)
    print(f"  {inf*100:>9.0f}% {fmt(infl_exp):>16s} {fmt(fn):>18s} {pct(rr):>14s}")

---

## §10 — Full 27-Year Trajectory Simulation

A phased simulation incorporating salary growth, major purchases, transport costs, rent, and shifting investment allocations across the full FIRE horizon.

In [ ]:
# ============================================================
# §10: Full 27-Year Trajectory Simulation
# ============================================================

def simulate_full_trajectory() -> List[dict]:
    """Year-by-year simulation mirroring app.js projectNetWorth()."""
    trajectory = []
    net_worth = CURRENT_SAVINGS
    invest_return = NOMINAL_RETURN
    savings_rate_pct = SAVINGS_RATE
    airtime = 1_000  # monthly
    
    for year in range(2026, 2053):
        age = year - 2002  # birth year
        
        # Phase-based salary and rent
        if year == 2026:
            salary, rent = STIPEND, 0
        elif year <= 2027:
            salary, rent = JOB_SALARY, 0
        elif year <= 2029:
            salary, rent = 100_000, 0
        elif year <= 2034:
            salary, rent = 150_000, 40_000
        elif year <= 2039:
            salary, rent = 250_000, 60_000
        elif year <= 2044:
            salary, rent = 350_000, 0  # property purchased
        else:
            salary, rent = 450_000, 0
        
        annual_contrib = (salary - rent) * savings_rate_pct * 12
        net_worth = net_worth * (1 + invest_return) + annual_contrib - (airtime * 12)
        
        # Major purchases
        if year == 2026:
            net_worth -= (6_995 + 80_000)  # suit + phone
        if year == 2030:
            net_worth -= 1_250_000  # car
        
        net_worth = max(0, net_worth)
        
        trajectory.append({
            "year": year,
            "age": age,
            "salary": salary,
            "rent": rent,
            "net_worth": round(net_worth),
        })
    
    return trajectory


print("=" * 60)
print("§10: FULL 27-YEAR TRAJECTORY")
print("=" * 60)

traj = simulate_full_trajectory()

print(f"{'Year':>6s} {'Age':>4s} {'Salary':>12s} {'Rent':>10s} {'Net Worth':>18s}")
print("-" * 52)
for row in traj:
    marker = " ★" if row["year"] in [2026, 2028, 2030, 2040, 2052] else ""
    print(f"{row['year']:>6d} {row['age']:>4d} {fmt(row['salary']):>12s}"
          f" {fmt(row['rent']):>10s} {fmt(row['net_worth']):>18s}{marker}")

final_nw = traj[-1]["net_worth"]
print(f"\nFinal net worth at age 50: {fmt(final_nw)}")
print(f"FIRE target:               {fmt(FIRE_TARGET)}")
gap = FIRE_TARGET - final_nw
if gap > 0:
    print(f"Shortfall:                 {fmt(gap)} ({gap/FIRE_TARGET*100:.1f}% of target)")
else:
    print(f"Surplus:                   {fmt(-gap)} (exceeded by {-gap/FIRE_TARGET*100:.1f}%)")

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
print(f"All formulae derived, validated, and cross-referenced with app.js.")
print(f"For interactive exploration, run the dashboard: python -m http.server 8080")